In [1]:
import MDAnalysis as mda
from numpy import *
import os
from pylab import *
import MDAnalysis.analysis.distances
import MDAnalysis.analysis.rms
from MDAnalysis.analysis import align
import glob
#import umap
import scipy.stats
import sklearn
import sklearn.decomposition
import sklearn.preprocessing
import pandas as pd
import seaborn as sns
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA
import mdtraj

In [2]:
import os

########################################################
#############      FOR NOW EQPOINT IS 0   ##############
########################################################
EQPOINT=0

systemFolders = glob.glob("huNumbering/*t5a*/")

systemgros=[]
systemtprs=[]
systemtrjs=[]
for i in range(len(systemFolders)):
    systemgros.append(sorted(glob.glob(systemFolders[i]+"*.gro")))
    systemtprs.append(sorted(glob.glob(systemFolders[i]+"*.tpr")))
    systemtrjs.append(sorted(glob.glob(systemFolders[i]+"*.xtc")))


    
    
threeColor=["#FE6100","#332288","#882255"]
colourScheme= threeColor
system_names = ["rhT5A","T5A","T5AR332P"]
systems=[]
for i in range(len(systemgros)):
    sub=[]
    for j in range(len(systemgros[i])):
        # When using TPRs, residues are indexed from 1; so we need to add in the first residue, 1 - 1 + first resid=first resid
        #firstres = mda.Universe(systemgros[i][j]).residues.resids[0]-1
        tu = mda.Universe(systemgros[i][j],systemtrjs[i][j])
        #tu.residues.resids +=firstres
                          
        sub.append(tu)
        
    systems.append(sub)

rhresids = arange(292,496)
huresids = systems[1][0].residues.resids
    
protein=[]
proteinstrings=[]
for i in range(len(systems)):
    sub = []
    sub2 = []
    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and not name H*"))
        sub2.append("protein and not name H*")
        
    protein.append(sub)
    proteinstrings.append(sub2)


bodys=[]
bodystrings=[]
# rhesus v1 is 326-349
# Human v1 is 324-345
#huloopstring = "resid 324:345"
#rhloopstring = "resid 326:349"
combinedLoopString="resid 324:345"
#system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    sub2 = []
    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and not ("+combinedLoopString+")"))
        sub2.append("protein and not ("+combinedLoopString+")")
        
    bodys.append(sub)
    bodystrings.append(sub2)
    
    
v1s=[]
v1strings=[]
# rhesus v1 is 326-349
# Human v1 is 324-345
#huloopstring = "resid 324:345"
#rhloopstring = "resid 326:349"
combinedLoopString="resid 324:345"
#system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    sub2 = []

    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and "+combinedLoopString))
        sub2.append("protein and "+combinedLoopString)

        
    v1s.append(sub)        
    v1strings.append(sub2)
                   

C:\Users\Liam\anaconda3\lib\site-packages\MDAnalysis\topology\base.py:203: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  residx = np.zeros_like(criteria[0], dtype=np.int)
C:\Users\Liam\anaconda3\lib\site-packages\MDAnalysis\core\selection.py:521: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: h

In [3]:
import numpy as np

def contact_size(system):
    '''The function calculates the size of the vector needed to contain contact map '''
    counter = 0
    for res1 in system.residues:
        for res2 in system.residues:
            if abs(res1.resid - res2.resid) > 1 and res1.resid < res2.resid:
                counter += 1
    return counter
def contact_map2(system, stride,sel="protein*",cutoff = 3.5):
    ''' Faster!!! The function creates contact map (0 and 1) for for every pair of aminoacids'''
    sel_heavy = sel
    num_data_points = (len(system.trajectory) // stride) + (0 if (len(system.trajectory) % stride == 0) else 1)
    size = contact_size(system.select_atoms(sel))
    contacts = np.empty((num_data_points, size), dtype=np.float)
    slices = []
    min_i = 0
    max_i = 0
    for res in system.select_atoms(sel).residues:
        max_i = min_i + res.atoms.select_atoms(sel_heavy).n_atoms
        slices.append((min_i, max_i))
        min_i = max_i

    #print(slices)
    heavy_system = system.select_atoms(sel_heavy)
    index = 0
    for ts in system.trajectory[::stride]:
        if ts.frame % 100 == 0 :
            output = round(100 * ts.frame / len(system.trajectory), 1)
            print(output,"% complete",end='\r')
            sys.stdout.flush()
        contact_map = np.zeros(size, dtype=np.float)
        contact_matrix = MDAnalysis.analysis.distances.contact_matrix(heavy_system.atoms.positions, cutoff = cutoff)
        counter = 0
        for i, res1 in enumerate(slices):
            i1 = res1[0]
            i2 = res1[1]
            for j, res2 in enumerate(slices):
                if abs(i - j) > 1 and i < j:
                    j1 = res2[0]
                    j2 = res2[1]
                    if np.any(contact_matrix[i1:i2, j1:j2]):
                        contact_map[counter] = 1.
                    counter += 1
        contacts[index,:] = contact_map
        index += 1
    return contacts


def getContacts(systems,datasets,selection_strings,stride=10,EQPOINT=250,cutoff = 3.5):
    
    contacts=[]
    stderrs=[]
    allcontacts=[]
    for i in range(len(datasets)):
        subcontacts=[]
        for j in range(len(datasets[i])):
            cont = contact_map2(systems[i][j],stride,sel = selection_strings[i][j],cutoff = cutoff)
            subcontacts.append(cont)
            #print(cont.shape)
        print(subcontacts)
        #contacts.append(mean(mean(subcontacts,axis = 0)[EQPOINT:],axis=0))
        allcontacts.append(subcontacts)
    
    return allcontacts

allcontacts_ts=getContacts(systems,protein,proteinstrings,stride = 1,EQPOINT=EQPOINT,cutoff = 4.5)# let's jsut mean along time as well

<ipython-input-3-e13693de14e0>:16: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  contacts = np.empty((num_data_points, size), dtype=np.float)
<ipython-input-3-e13693de14e0>:33: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  contact_map = np.zeros(size, dtype=np.float)
C:\Users\Liam\anaconda3\lib\site-packages\MDAnalysis\lib\distances.py:71

[array([[1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.],
       ...,
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.]]), array([[1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 0., 0.],
       ...,
       [1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.]]), array([[1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 1., 0.],
       ...,
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.]]), array([[1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.],
       ...,
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 1., 0.]]), array([[1., 1., 1., ..., 0., 1.

In [4]:
#save("allcontacts_mean.npy",allcontacts)
for i in range(len(system_names)):
    save("allcontacts_ts_"+system_names[i]+"_4d5.npy",allcontacts_ts[i])

In [5]:
#allcontacts_ts=[]
#for i in range(len(system_names)):
#    allcontacts_ts.append(load("allcontacts_ts_"+system_names[i]+"4d5.npy"))

FileNotFoundError: [Errno 2] No such file or directory: 'allcontacts_ts_rhT5A4d5.npy'

In [ ]:
#for i in range(len(system_names)):
#    save("allcontacts_ts_"+system_names[i]+"_stride2_4d5.npy",allcontacts_ts[i][:,::2,:])